# 🤖 RAG com LLM — Chatbot para Análise de Documentos Financeiros
**Retrieval Augmented Generation com Google Gemini + Relatório de Estabilidade Financeira do Banco Central**

---

## 🎯 Objetivo

Construir um sistema de **RAG (Retrieval Augmented Generation)** capaz de responder perguntas em linguagem natural sobre documentos financeiros complexos.

O sistema extrai o texto do PDF, divide em chunks, e usa o Google Gemini para responder perguntas com base no conteúdo real do documento — sem alucinar informações.

---

## 0. Imports e Configurações

In [ ]:
import google.generativeai as genai
import PyPDF2
import os
from dotenv import load_dotenv
import textwrap
import time

# Carrega a API key do .env
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY')

if not api_key:
    raise ValueError('❌ GEMINI_API_KEY não encontrada no .env!')

genai.configure(api_key=api_key)
print('✅ API configurada com sucesso!')
print(f'🔑 Key carregada: {api_key[:10]}...')

## 1. Extração do Texto do PDF

In [ ]:
PDF_PATH = r'C:\Users\roney\OneDrive\Área de Trabalho\Projetos\Projeto_LLM\RELESTAB202510-refPub.pdf'

def extrair_texto_pdf(caminho):
    """Extrai todo o texto de um PDF página por página."""
    texto_completo = []
    
    with open(caminho, 'rb') as arquivo:
        leitor = PyPDF2.PdfReader(arquivo)
        total_paginas = len(leitor.pages)
        print(f'📄 Total de páginas: {total_paginas}')
        
        for i, pagina in enumerate(leitor.pages):
            texto = pagina.extract_text()
            if texto and texto.strip():
                texto_completo.append({
                    'pagina': i + 1,
                    'texto': texto.strip()
                })
    
    return texto_completo

paginas = extrair_texto_pdf(PDF_PATH)
print(f'✅ {len(paginas)} páginas com conteúdo extraídas')
print(f'\n📖 Prévia da página 1:')
print(paginas[0]['texto'][:500] + '...')

## 2. Divisão em Chunks

Dividimos o texto em pedaços menores (chunks) para que o LLM consiga processar e encontrar a informação relevante com mais precisão.

In [ ]:
def criar_chunks(paginas, tamanho_chunk=3000, sobreposicao=200):
    """
    Divide o texto em chunks com sobreposição para não perder contexto.
    tamanho_chunk: caracteres por chunk
    sobreposicao: caracteres de overlap entre chunks consecutivos
    """
    texto_total = '\n\n'.join([p['texto'] for p in paginas])
    chunks = []
    inicio = 0
    
    while inicio < len(texto_total):
        fim = inicio + tamanho_chunk
        chunk = texto_total[inicio:fim]
        
        # Tenta quebrar no final de uma frase
        if fim < len(texto_total):
            ultimo_ponto = chunk.rfind('.')
            if ultimo_ponto > tamanho_chunk * 0.7:
                chunk = chunk[:ultimo_ponto + 1]
                fim = inicio + ultimo_ponto + 1
        
        chunks.append(chunk)
        inicio = fim - sobreposicao
    
    return chunks

chunks = criar_chunks(paginas)
print(f'✅ {len(chunks)} chunks criados')
print(f'📏 Tamanho médio: {sum(len(c) for c in chunks) // len(chunks)} caracteres/chunk')
print(f'\n📋 Prévia do chunk 1:')
print(chunks[0][:300] + '...')

## 3. Busca por Relevância

Quando o usuário faz uma pergunta, precisamos encontrar os chunks mais relevantes antes de enviar ao LLM — isso reduz custo e melhora a precisão.

In [ ]:
def buscar_chunks_relevantes(pergunta, chunks, top_k=3):
    """
    Busca simples por palavras-chave para encontrar chunks relevantes.
    Em produção, usaríamos embeddings vetoriais (ex: FAISS + sentence-transformers).
    """
    pergunta_lower = pergunta.lower()
    palavras = [p for p in pergunta_lower.split() if len(p) > 3]
    
    scores = []
    for i, chunk in enumerate(chunks):
        chunk_lower = chunk.lower()
        score = sum(1 for palavra in palavras if palavra in chunk_lower)
        scores.append((score, i, chunk))
    
    scores.sort(reverse=True)
    return [chunk for _, _, chunk in scores[:top_k]]

# Teste da busca
pergunta_teste = "Qual é a situação do sistema financeiro?"
chunks_relevantes = buscar_chunks_relevantes(pergunta_teste, chunks)
print(f'🔍 Busca por: "{pergunta_teste}"')
print(f'✅ {len(chunks_relevantes)} chunks relevantes encontrados')
print(f'\nChunk mais relevante (prévia):')
print(chunks_relevantes[0][:400] + '...')

## 4. Sistema RAG — Integração com Gemini

In [ ]:
modelo = genai.GenerativeModel('gemini-1.5-flash')

def responder_pergunta(pergunta, chunks, verbose=True):
    """
    Pipeline RAG completo:
    1. Busca chunks relevantes
    2. Monta prompt com contexto
    3. Envia ao Gemini
    4. Retorna resposta
    """
    # Passo 1: Recuperar contexto relevante
    chunks_relevantes = buscar_chunks_relevantes(pergunta, chunks, top_k=3)
    contexto = '\n\n---\n\n'.join(chunks_relevantes)
    
    # Passo 2: Montar prompt
    prompt = f"""Você é um analista financeiro especializado em relatórios do Banco Central do Brasil.
Responda a pergunta abaixo com base EXCLUSIVAMENTE no contexto fornecido.
Se a informação não estiver no contexto, diga claramente que não encontrou essa informação no documento.
Responda sempre em português brasileiro de forma clara e objetiva.

CONTEXTO DO DOCUMENTO:
{contexto}

PERGUNTA: {pergunta}

RESPOSTA:"""
    
    # Passo 3: Chamar o Gemini
    resposta = modelo.generate_content(prompt)
    
    if verbose:
        print(f'\n❓ Pergunta: {pergunta}')
        print(f'\n🤖 Resposta do Gemini:')
        print('-' * 60)
        print(textwrap.fill(resposta.text, width=80))
        print('-' * 60)
    
    return resposta.text

print('✅ Sistema RAG configurado!')
print('🤖 Modelo: gemini-1.5-flash')

## 5. Testando o Sistema com Perguntas Reais

In [ ]:
# Pergunta 1
resposta1 = responder_pergunta(
    "Qual é a avaliação geral sobre a estabilidade do sistema financeiro brasileiro?",
    chunks
)

In [ ]:
time.sleep(2)  # Respeita o rate limit da API gratuita

# Pergunta 2
resposta2 = responder_pergunta(
    "Quais são os principais riscos identificados para o sistema financeiro?",
    chunks
)

In [ ]:
time.sleep(2)

# Pergunta 3
resposta3 = responder_pergunta(
    "Como está o nível de capitalização dos bancos brasileiros?",
    chunks
)

In [ ]:
time.sleep(2)

# Pergunta 4
resposta4 = responder_pergunta(
    "Qual é a situação da inadimplência no crédito?",
    chunks
)

## 6. Chatbot Interativo

In [ ]:
def chatbot_interativo(chunks):
    """
    Loop interativo para fazer perguntas ao documento.
    Digite 'sair' para encerrar.
    """
    print('🤖 Chatbot RAG — Relatório de Estabilidade Financeira (BCB)')
    print('=' * 60)
    print('Digite sua pergunta sobre o documento ou "sair" para encerrar.')
    print('=' * 60)
    
    historico = []
    
    while True:
        pergunta = input('\n❓ Você: ').strip()
        
        if pergunta.lower() in ['sair', 'exit', 'quit']:
            print('\n👋 Encerrando chatbot. Até logo!')
            break
        
        if not pergunta:
            print('Por favor, digite uma pergunta.')
            continue
        
        resposta = responder_pergunta(pergunta, chunks, verbose=False)
        historico.append({'pergunta': pergunta, 'resposta': resposta})
        
        print(f'\n🤖 Assistente:')
        print('-' * 60)
        print(textwrap.fill(resposta, width=80))
        print('-' * 60)
        
        time.sleep(1)  # Rate limit
    
    return historico

# Descomente para usar o chatbot interativo:
# historico = chatbot_interativo(chunks)
print('💡 Para usar o chatbot interativo, descomente a última linha desta célula.')

---
## 📋 Como Funciona o RAG

```
📄 PDF                    🔍 Busca                    🤖 LLM
─────────────────────────────────────────────────────────────
Documento  →  Chunks  →  Chunks         →  Prompt    →  Resposta
           →  (texto      relevantes    →  (contexto    fundamentada
               dividido)  (por keyword)    + pergunta)  no documento
```

## 🚀 Próximos Passos

- **Embeddings vetoriais:** Substituir a busca por keyword por embeddings semânticos (FAISS + sentence-transformers) para resultados mais precisos
- **Memória de conversa:** Manter histórico para perguntas de follow-up
- **Interface web:** Criar um frontend com Streamlit para uso sem código
- **Múltiplos documentos:** Expandir para indexar vários PDFs simultaneamente